# Análise Descritiva - Capítulo 44

Este notebook contém uma análise descritiva sobre os dados de itens de nota fiscal do Capítulo 44 do NCM, conforme dados disponibilizados em ```tb_treina_cap44.csv```

## 1. Configuração Inicial e Carregamento de Dados (Pandas)

In [ ]:
import pandas as pd
import re
import os
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from IPython.display import display
from gensim.models import Word2Vec, FastText

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

FILE_PATH = 'cap_44.csv'

# O arquivo é separado por tabulação ('\t')
try:
    df_pandas = pd.read_csv(FILE_PATH, sep='\t', decimal='.')
    print(f"Total de linhas: {len(df_pandas)}")
    
    df_pandas['prod_vuntrib'] = pd.to_numeric(df_pandas['prod_vuntrib'], errors='coerce')
    df_pandas['prod_qtrib'] = pd.to_numeric(df_pandas['prod_qtrib'], errors='coerce')
    df_pandas['contagem'] = pd.to_numeric(df_pandas['contagem'], errors='coerce', downcast='integer')
    
    df_pandas.info()
    print("\nPrimeiras 5 linhas:")
    print(df_pandas.head())
except Exception as e:
    print(f"Ocorreu um erro durante o carregamento do Pandas: {e}")

In [ ]:
# Normalização para MAIÚSCULAS
df_pandas['prod_xprod_norm'] = df_pandas['prod_xprod'].astype(str).str.upper()
print(df_pandas['prod_xprod_norm'].head())

## 2. Análise de Contagem (`contagem`)

In [ ]:
# Ordenar os produtos por contagem (qtd) (tanto asc como desc)
print("Top 50 Produtos por Contagem (Mais Frequentes)")
top_50_contagem = df_pandas.sort_values(by='contagem', ascending=False).head(50)
print(top_50_contagem[['prod_xprod', 'contagem', 'prod_vuntrib']].reset_index(drop=True))

In [ ]:
print("\nBottom 50 Produtos por Contagem (Menos Frequentes)")
bottom_50_contagem = df_pandas.sort_values(by='contagem', ascending=True).head(50)
print(bottom_50_contagem[['prod_xprod', 'contagem', 'prod_vuntrib']].reset_index(drop=True))

## 3. Análise de Preço (`prod_vuntrib` e `prod_qtrib`)

In [ ]:
print("Top 50 Produtos por Valor Unitário Tributável (prod_vuntrib)")
top_50_vuntrib = df_pandas.sort_values(by='prod_vuntrib', ascending=False).head(50)
print(top_50_vuntrib[['prod_xprod', 'prod_vuntrib', 'contagem']].reset_index(drop=True))

In [ ]:
print("Bottom 50 Produtos por Valor Unitário Tributável (prod_vuntrib)")
bottom_50_vuntrib = df_pandas.sort_values(by='prod_vuntrib', ascending=True).head(50)
print(bottom_50_vuntrib[['prod_xprod', 'prod_vuntrib', 'contagem']].reset_index(drop=True))

In [ ]:
print("Top 50 Produtos por Quantidade Tributável (prod_qtrib)")
top_50_qtrib = df_pandas.sort_values(by='prod_qtrib', ascending=False).head(50)
print(top_50_qtrib[['prod_xprod', 'prod_qtrib', 'contagem']].reset_index(drop=True))

In [ ]:
print("Bottom 50 Produtos por Quantidade Tributável (prod_qtrib)")
bottom_50_qtrib = df_pandas.sort_values(by='prod_qtrib', ascending=True).head(50)
print(bottom_50_qtrib[['prod_xprod', 'prod_qtrib', 'contagem']].reset_index(drop=True))

## 4. Análise de Representatividade (Valor Total)

In [ ]:
# Criar colunas de representatividade (Valor Total = Valor Unitário * Contagem)
df_pandas['valor_total_vuntrib'] = df_pandas['prod_vuntrib'] * df_pandas['contagem']
df_pandas['valor_total_qtrib'] = df_pandas['prod_qtrib'] * df_pandas['contagem']

In [ ]:
print("Top 50 Produtos por Valor Total Tributável (valor_total_vuntrib)")
top_50_total_vuntrib = df_pandas.sort_values(by='valor_total_vuntrib', ascending=False).head(50)
print(top_50_total_vuntrib[['prod_xprod', 'prod_vuntrib', 'contagem', 'valor_total_vuntrib']].reset_index(drop=True))

In [ ]:
print("Bottom 50 Produtos por Valor Total Tributável (valor_total_vuntrib)")
bottom_50_total_vuntrib = df_pandas.sort_values(by='valor_total_vuntrib', ascending=True).head(50)
print(bottom_50_total_vuntrib[['prod_xprod', 'prod_vuntrib', 'contagem', 'valor_total_vuntrib']].reset_index(drop=True))

In [ ]:
print("Top 50 Produtos por Quantidade Total Tributável (valor_total_qtrib)")
top_50_total_qtrib = df_pandas.sort_values(by='valor_total_qtrib', ascending=False).head(50)
print(top_50_total_qtrib[['prod_xprod', 'prod_qtrib', 'contagem', 'valor_total_qtrib']].reset_index(drop=True))

In [ ]:
print("Bottom 50 Produtos por Quantidade Total Tributável (valor_total_qtrib)")
bottom_50_total_qtrib = df_pandas.sort_values(by='valor_total_qtrib', ascending=True).head(50)
print(bottom_50_total_qtrib[['prod_xprod', 'prod_qtrib', 'contagem', 'valor_total_qtrib']].reset_index(drop=True))

## Regex

In [ ]:
# Termos de exclusão para as regras complexas
termos_excl_janela = 'ALUMINIO|FERRO|METAL|INOX|PVC|VIDRO|BLINDEX|FUME|ACRILICO|PLASTICO|APLIQUE|RECORTE|LASER|MINIATURA|BONECA|POLLY|BRINQUEDO|ARTESANATO|ENFEITE|DECORACAO|FERRAGEM|FECHADURA|DOBRADICA|TRILHO|PUXADOR|CREMONA|ESPUMA|PU|VEDACAO|IMA|GELADEIRA|CORTINA|PERSIANA|TELA|MOSQUITEIRO|ADESIVO|LIMPEZA|PET|GATO|CACHORRO|ALIMENTADOR|ALBUM|FOTO|FICHARIO|HARD CASE|BAMBU|GIRASSOL|VASO|FLOR|PASSARO|CENARIO|ALTAR|CAPELA|ESPELHO|BASTIDOR|CACHEPO|PORTA RETRATO|JOIA|PRESENTE|ORGANIZADOR|TAMPO|FOGAO|MESA|CADEIRA'
termos_excl_carvao = 'ATIVADO|ATIVO|MINERAL|ANTRACIT|COQUE|HULHA|LINHITA|TURFA|FILTRO|MASCARA|SHAMPOO|SABONETE|CAPSULA|COMPRIMIDO|DENTIFRICO|PASTA DE DENTE'
termos_excl_palete = 'LOCACAO|REFORMA|ALUGUEL|CONSERTO|FRETE|MANUTENCAO|REPARO'
termos_excl_mdf = 'ARMARIO|ESTANTE|GAVETEIRO|MESA|CADEIRA|BALCAO|CRIADO|GUARDA ROUPA'
termos_ambiguos_exclusao = 'PLANEJADA|PLANEJADOS|COZINHA|DALMOBILE|EIXO|GUINDASTE|ELEVADOR|AUTOMATIC|ELETROMECANICO|ELETROHIDRAULICO|VW|DEERE|JOHN|CABINES|BOVINA|BOVINOS|CONTENCAO|PESCOCEIRAS|COIMMA|CAPUCCINO|FAVOS|IMPRESSO'
termos_ambiguos_inclusao = 'CABO|CABOS|FOLHA|FERRAMENTAS'

# Categorias REGEX
categorias_regex = [
    ('ROTULADOS ANTES', r'(LENHA|TORA DE MADEIRA|MADEIRA CERRADA|CABO DE MADEIRA|ARTIGO DOMESTICO DE MADEIRA|CARVAO VEGETAL|PORTA-BATENTE DE MADEIRA|PALITO DE DENTE|PALLET-EMBALAGEM DE MADEIRA|CHAPAS DE MDF|CHAPAS DE COMPENSADO|CHAPAS DE TAPUME|RESIDUO DE MADEIRA)'),
    ('OUTLIER__MAQUINAS', r'(PICADOR|TRATOR|COLHEITADEIRA|MÁQUINA|MAQUINA|VEÍCULO|VEICULO|PICADOR FLORESTAL)'),
    ('44.18.JANELA_MADEIRA', rf'^(?=.*JANELA)(?!.*({termos_excl_janela})).*'),
    ('44.02.CARVAO_VEGETAL', rf'^(?=.*(CARVÃO|CARVAO|MOINHA))(?!.*({termos_excl_carvao})).*'),
    ('44.21.PALETE_PALLET', rf'^(?=.*(PALETE|PALLET))(?!.*({termos_excl_palete})).*'),
    ('44.11.MDF_PAINEL', rf'^(?=.*(MDF|MDP|OSB|CHAPATEX|HDF))(?!.*({termos_excl_mdf})).*'),
    ('AMBIGUOS_EXCLUSAO', rf'.*({termos_ambiguos_exclusao}).*'),
    ('NOMES_AMBÍGUOS', rf'^(?=.*({termos_ambiguos_inclusao}))(?=.*(MADEIRA|MDF|MDP|PINUS|MM|X)).*'),
    ('44.03.MADEIRA_BRUTA', r'(MADEIRA.*(BRUTA|TORA|CELULOSE)|EUCALIPTO.*(TORA|CELULOSE)|PINUS.*BRUTA)'),
    ('44.18.FOLHA_PORTA', r'(FOLHA DE PORTA|PORTA|BATENTE|CAIXILHO|FORRO|SOALHO|PISO|ASSOALHO)'),
    ('44.01.CAVACO_RESIDUO', r'(CAVACO DE EUCALIPTO|LENHA|CAVACO|RESIDUO|GRANULADO|BIOMASSA|SERRAGEM|PO DE MADEIRA)'),
    ('44.21.OUTRAS_OBRAS', r'(PALITO|CEPO|ESTRADO|CAIXOTE)'),
    ('OUTROS', r'.*')
]

def categorizar_produto(descricao):
    descricao_upper = str(descricao).upper()
    for categoria, regex in categorias_regex:
        if re.search(regex, descricao_upper):
            return categoria
    return 'NAO_CLASSIFICADO'

df_pandas['valor_total_vuntrib'] = df_pandas['prod_vuntrib'] * df_pandas['prod_qtrib'] * df_pandas['contagem']
df_pandas['categoria_regex'] = df_pandas['prod_xprod'].apply(categorizar_produto)

In [ ]:
categorias_cap_44 = [
    'ROTULADOS ANTES', '44.18.JANELA_MADEIRA', '44.02.CARVAO_VEGETAL', 
    '44.21.PALETE_PALLET', '44.11.MDF_PAINEL', 'NOMES_AMBÍGUOS', 
    '44.03.MADEIRA_BRUTA', '44.18.FOLHA_PORTA', '44.01.CAVACO_RESIDUO', 
    '44.21.OUTRAS_OBRAS'
]
categorias_eliminadas = ['OUTLIER__MAQUINAS', 'AMBIGUOS_EXCLUSAO']

df_cap_44 = df_pandas[df_pandas['categoria_regex'].isin(categorias_cap_44)].copy()
df_outros = df_pandas[df_pandas['categoria_regex'] == 'OUTROS'].copy()
df_eliminados = df_pandas[df_pandas['categoria_regex'].isin(categorias_eliminadas)].copy()

df_cap_44.to_csv('base_treino_capitulo_44.csv', sep=';', index=False, encoding='utf-8-sig')
df_outros.to_csv('base_outros_a_analisar.csv', sep=';', index=False, encoding='utf-8-sig')
df_eliminados.to_csv('base_eliminados_lixo.csv', sep=';', index=False, encoding='utf-8-sig')

print(f"1. CAPÍTULO 44: {len(df_cap_44):>10,}".replace(",", "."))
print(f"2. OUTROS:        {len(df_outros):>10,}".replace(",", "."))
print(f"3. ELIMINADOS (Lixo/Outros Cap): {len(df_eliminados):>10,}".replace(",", "."))
print(f"TOTAL DE ITENS PROCESSADOS:      {len(df_pandas):>10,}".replace(",", "."))


In [ ]:
print(" Distribuição de Produtos por Categoria Regex (Top 20)")
distribuicao_regex = df_pandas['categoria_regex'].value_counts().head(20)
print(distribuicao_regex)

print(" Representatividade Financeira por Categoria (Top 20)")
representatividade_regex = (
    df_pandas.groupby('categoria_regex')['valor_total_vuntrib']
    .sum()
    .sort_values(ascending=False)
)

total = representatividade_regex.sum()
representatividade_percentual = (representatividade_regex / total) * 100
print(representatividade_percentual.head(20).round(2).astype(str) +"%")

In [ ]:
# 1. Representatividade por CONTAGEM (Volume de Dados)
contagem_absoluta = df_pandas['categoria_regex'].value_counts()
total_linhas = len(df_pandas)

representatividade_contagem_pct = (contagem_absoluta / total_linhas) * 100
print(representatividade_contagem_pct.head(20).round(2).astype(str) +"%")

# 2. Comparação Direta (Contagem vs Financeiro) 
comparativo = pd.DataFrame({
    'Volume (%)': representatividade_contagem_pct,
    'Financeiro (%)': representatividade_percentual
})
print(comparativo.head(20).round(2))


## 6. Para análises específicas

In [ ]:
df_pandas.head()

In [ ]:
'''ITEM ="MDF"
base ="cap_44.csv"
mask = df_pandas.apply(lambda col: col.astype(str).str.contains(ITEM, case=False, na=False))
df_filtrado = df_pandas[mask.any(axis=1)]
arquivo_saida = f"{ITEM}.csv"
df_filtrado.to_csv(arquivo_saida, index=False)'''

In [ ]:
df_outros = df_pandas[df_pandas['categoria_regex'] =="OUTROS"]
arquivo_saida ="OUTROS.csv"
df_outros.to_csv(arquivo_saida, index=False)

In [ ]:
# Criar um DataFrame apenas com as colunas necessárias para validação
df_validacao = df_pandas[['prod_xprod', 'categoria_regex']].copy()

# Salvar em CSV para conferência manual
# Usamos sep='\t'
df_validacao.to_csv('validacao_categorias_ncm44.csv', sep='\t', index=False, encoding='utf-8-sig')

print("Arquivo 'validacao_categorias_ncm44.csv' gerado com sucesso!")
# Mostrar uma amostra rápida de cada categoria
for cat in df_validacao['categoria_regex'].unique():
    print(f" Amostra da Categoria: {cat}")
    print(df_validacao[df_validacao['categoria_regex'] == cat]['prod_xprod'].head(3).tolist())


 
## 7. Novas Ferramentas de NLP (BoW, TF-IDF e Embeddings)

### 2. Bag of Words (BoW)
Representação de contagem de palavras para o dataset principal e filtros específicos.

In [ ]:
# 2.1 Bag of Words com Visualização
print("Processando Bag of Words...")
vectorizer_bow = CountVectorizer(max_features=1000, stop_words=['DE', 'DA', 'DO', 'COM', 'PARA', 'EM'])
bow_matrix = vectorizer_bow.fit_transform(df_pandas['prod_xprod_norm'])

# Criando um DataFrame para visualizar os Top 30 termos
sum_words = bow_matrix.sum(axis=0)
words_freq = [(word, sum_words[0, idx]) for word, idx in vectorizer_bow.vocabulary_.items()]
words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)

df_bow_viz = pd.DataFrame(words_freq, columns=['Termo', 'Frequência'])
print("\nTop 30 Termos - Bag of Words:")
display(df_bow_viz.head(30))

### 3. TF-IDF
Representação de frequência inversa de documentos para o dataset principal e filtros específicos.

In [ ]:
# 3.1 TF-IDF com Visualização
print("Processando TF-IDF...")

vectorizer_tfidf = TfidfVectorizer(
    max_features=1000, 
    max_df=0.7,  # IGNORE termos que aparecem em mais de 70% dos itens
    min_df=5,    # IGNORE termos que aparecem em menos de 5 itens
    ngram_range=(1, 2) # Captura"MADEIRA PINUS"como um único termo
)
tfidf_matrix = vectorizer_tfidf.fit_transform(df_pandas['prod_xprod_norm'])

# Criando um DataFrame para visualizar os Top 30 termos por Score Médio
feature_names = vectorizer_tfidf.get_feature_names_out()
avg_tfidf = tfidf_matrix.mean(axis=0).tolist()[0]
tfidf_scores = sorted(list(zip(feature_names, avg_tfidf)), key=lambda x: x[1], reverse=True)

df_tfidf_viz = pd.DataFrame(tfidf_scores, columns=['Termo', 'Score_TFIDF'])
print("\nTop 30 Termos - TF-IDF (Importância Relativa):")
display(df_tfidf_viz.head(30))

### 4. Word Embeddings
Implementação de Word2Vec e FastText.

In [ ]:
def simple_tokenize(text):
    return re.findall(r'\w+', str(text).upper())

sentences = df_pandas['prod_xprod_norm'].apply(simple_tokenize).tolist()
print(f"Tokenização: {len(sentences)} sentenças preparadas.")

# 4.1 Word2Vec
print("Treinando Word2Vec")
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=5, workers=4)

termo_teste = 'MADEIRA'
if termo_teste in w2v_model.wv:
    print(f"\nPalavras mais similares a '{termo_teste}':")
    display(pd.DataFrame(w2v_model.wv.most_similar(termo_teste, topn=20), columns=['Termo', 'Similaridade']))

In [ ]:
# 4.2 FastText - Treinamento
print("Treinando FastText (lidando com sub-palavras e erros de digitação)...")
ft_model = FastText(sentences, vector_size=100, window=5, min_count=5, workers=4)
print("FastText treinado com sucesso!")

termo_teste_ft = 'MADEIRA'
if termo_teste_ft in ft_model.wv:
    print(f"\nPalavras mais similares a '{termo_teste_ft}' (via FastText):")
    display(pd.DataFrame(ft_model.wv.most_similar(termo_teste_ft, topn=30), columns=['Termo', 'Similaridade']))

In [ ]:
# 4.2 FastText - Treinamento
print("Treinando FastText (lidando com sub-palavras e erros de digitação)...")
ft_model = FastText(sentences, vector_size=100, window=5, min_count=5, workers=4)
print("FastText treinado com sucesso!")

termo_teste_ft = 'JANELA'
if termo_teste_ft in ft_model.wv:
    print(f"\nPalavras mais similares a '{termo_teste_ft}' (via FastText):")
    display(pd.DataFrame(ft_model.wv.most_similar(termo_teste_ft, topn=30), columns=['Termo', 'Similaridade']))

## Exportação de dados financeiros para análise

In [ ]:
df = df_pandas

# 2. NORMALIZAÇÃO E LIMPEZA
df['prod_xprod_norm'] = df['prod_xprod'].astype(str).str.upper()
df['prod_vuntrib'] = pd.to_numeric(df['prod_vuntrib'], errors='coerce').fillna(0)
df['prod_qtrib'] = pd.to_numeric(df['prod_qtrib'], errors='coerce').fillna(0)
df['contagem'] = pd.to_numeric(df['contagem'], errors='coerce').fillna(1)

# Calcular valor total por linha (Valor Unitário * Quantidade de Ocorrências)
df['valor_total_vuntrib'] = df['prod_vuntrib'] * df['contagem']
df['valor_total_qtrib'] = df['prod_qtrib'] * df['contagem']

# Stopwords a ignorar (Palavras irrelevantes para a análise financeira)
stopwords = {
    'DE', 'DA', 'DO', 'COM', 'PARA', 'EM', 'UM', 'UMA', 'CM', 'MM', 'MT', 
    'KG', 'UN', 'UND', 'C/', 'X', 'O', 'A', 'OS', 'AS', 'PARA', 'POR', 'NAS'
}

# Dicionário para acumular estatísticas: {palavra: [soma_vuntrib, soma_qtrib, contagem_ocorrencias]}
word_stats = defaultdict(lambda: [0.0, 0.0, 0])

# 3. PROCESSAMENTO POR LINHA
for idx, row in df.iterrows():
    # Tokenização: extrai apenas palavras (letras e números)
    words = set(re.findall(r'\w+', row['prod_xprod_norm']))
    v_total = row['valor_total_vuntrib']
    q_total = row['valor_total_qtrib']
    
    for word in words:
        # Filtra stopwords, números puros e palavras muito curtas
        if word not in stopwords and not word.isdigit() and len(word) > 1:
            stats = word_stats[word]
            stats[0] += v_total
            stats[1] += q_total
            stats[2] += 1

# 4. CONSOLIDAÇÃO DOS RESULTADOS
result_data = []
for word, stats in word_stats.items():
    result_data.append({
        'Termo': word,
        'Valor_Total_Vuntrib': stats[0],
        'Valor_Total_Qtrib': stats[1],
        'Ocorrencias': stats[2],
        'Valor_Medio_Vuntrib': stats[0] / stats[2] if stats[2] > 0 else 0,
        'Valor_Medio_Qtrib': stats[1] / stats[2] if stats[2] > 0 else 0
    })

df_results = pd.DataFrame(result_data)

# 5. GERAÇÃO DOS TOP 50
# Nota: Para o valor médio, filtramos termos com menos de 10 ocorrências para evitar distorções por erros pontuais
top_50_total_vuntrib = df_results.sort_values(by='Valor_Total_Vuntrib', ascending=False).head(50)
top_50_medio_vuntrib = df_results[df_results['Ocorrencias'] > 10].sort_values(by='Valor_Medio_Vuntrib', ascending=False).head(50)
top_50_total_qtrib = df_results.sort_values(by='Valor_Total_Qtrib', ascending=False).head(50)
top_50_medio_qtrib = df_results[df_results['Ocorrencias'] > 10].sort_values(by='Valor_Medio_Qtrib', ascending=False).head(50)

# 6. SALVAMENTO DOS ARQUIVOS
top_50_total_vuntrib.to_csv('top_50_total_vuntrib.csv', index=False)
top_50_medio_vuntrib.to_csv('top_50_medio_vuntrib.csv', index=False)
top_50_total_qtrib.to_csv('top_50_total_qtrib.csv', index=False)
top_50_medio_qtrib.to_csv('top_50_medio_qtrib.csv', index=False)

print("Os 4 arquivos CSV com o Top 50 foram gerados.")


In [ ]:
def analisar_valor_transacao_por_termo(df):
    
    df['valor_transacao_total'] = df['prod_vuntrib'] * df['prod_qtrib'] * df['contagem']

    # Stopwords a ignorar
    stopwords = {
        'DE', 'DA', 'DO', 'COM', 'PARA', 'EM', 'UM', 'UMA', 'CM', 'MM', 'MT', 
        'KG', 'UN', 'UND', 'C/', 'X', 'O', 'A', 'OS', 'AS', 'POR', 'NAS', 'NOS',
        'E', 'OU', 'SE', 'NA', 'NO', 'NOS', 'NAS', 'AO', 'AOS', 'AS', 'ESTE', 'ESTA',
        'ESTES', 'ESTAS', 'ESSE', 'ESSA', 'ESSES', 'ESSAS', 'AQUELA', 'AQUELE', 'AQUELAS', 'AQUELOS'
    }

    word_stats = defaultdict(lambda: [0.0, 0])
    
    def processar_linha(row):
        words = set(re.findall(r'\w+', row['prod_xprod_norm']))
        v_transacao = row['valor_transacao_total']
        
        for word in words:
            # Filtros: Stopwords, dígitos e tamanho > 1
            if word not in stopwords and not word.isdigit() and len(word) > 1:
                stats = word_stats[word]
                stats[0] += v_transacao
                stats[1] += 1

    # Aplica a função de processamento linha a linha
    df.apply(processar_linha, axis=1)

    # 2. Consolidação
    result_data = []
    for word, stats in word_stats.items():
        result_data.append({
            'Termo': word,
            'Valor_Transacao_Total': stats[0],
            'Ocorrencias': stats[1],
            'Valor_Transacao_Medio': stats[0] / stats[1] if stats[1] > 0 else 0
        })

    df_results = pd.DataFrame(result_data)

    # 3. Geração dos Top 100
    print("Gerando arquivos de saída...")
    
    # Top 100 Total
    top_100_total = df_results.sort_values(by='Valor_Transacao_Total', ascending=False).head(100)
    top_100_total.to_csv('top_100_total_transacao_2.csv', index=False, encoding='utf-8-sig')
    
    # Top 100 Médio
    top_100_medio = df_results[df_results['Ocorrencias'] >= 10].sort_values(by='Valor_Transacao_Medio', ascending=False).head(100)
    top_100_medio.to_csv('top_100_medio_transacao_2.csv', index=False, encoding='utf-8-sig')

    print(f"Total de termos únicos analisados: {len(df_results)}")
    
    return df_results


In [ ]:
if 'df_pandas' in locals() and 'prod_xprod_norm' in df_pandas.columns:
    df_termos_analisados = analisar_valor_transacao_por_termo(df_pandas)
else:
    print("Erro: O DataFrame 'df_pandas' ou a coluna 'prod_xprod_norm' não foram encontrados. Certifique-se de executar as células anteriores.")